# Flood Simulation — Trier, Germany (14-Day HQ100 Event)

Simulates a complete 14-day riverine flood event with **hourly time steps** (336
frames total).  The flood extent is approximated by progressively eroding the
HQ100 polygon inward — see
`utils/flood_interpolation.py` for the algorithm.

| Phase | Hours | Days | Flood level |
|---|---|---|---|
| Pre-event | 0–47 | 1–2 | 0 % (no flooding) |
| Rising water | 48–143 | 3–6 | 0 % → 100 % |
| HQ100 peak | 144–191 | 7–8 | 100 % (full flood) |
| Receding | 192–287 | 9–12 | 100 % → 0 % |
| Post-event | 288–335 | 13–14 | 0 % (no flooding) |

Intermediate flood geometries are cached in
`data/processed/flood_interpolation/` and reused on subsequent runs.

**Keyboard shortcuts** (once the simulation is displayed):
`Space`/`K` — play/pause ·
`←`/`→` — step 1 h ·
`↑`/`↓` — step 1 d

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import geopandas as gpd
import osmnx as ox
from shapely.ops import unary_union
from IPython.display import HTML, display

from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import (
    RoaNotebookConfig,
    get_roa_cache_path,
    get_roa_hazard_data_path,
    get_roa_outputs_path,
    set_notbook_wd,
)
from css_geodata_service.robustness_of_accessibility.utils.flood_interpolation import (
    SIMULATION_HOURS,
    _RISE_START, _RISE_END, _PEAK_END, _FALL_END,
    build_flood_animation_html,
    compute_hourly_flood_progress,
    load_or_compute_flood_stages,
)

ox.settings.log_console = False
ox.settings.use_cache = True

print(f"osmnx     : {ox.__version__}")
print(f"geopandas : {gpd.__version__}")

osmnx     : 1.9.3
geopandas : 1.1.3


## 1. Configuration & Paths

In [2]:
set_notbook_wd()

place_name: str = RoaNotebookConfig.place_name   # "Trier, Germany"
event           = RoaNotebookConfig.event         # HQ100

cache_dir:       Path = get_roa_cache_path()
output_dir:      Path = get_roa_outputs_path()
hazard_data_path: Path = get_roa_hazard_data_path(event=event)

output_dir.mkdir(parents=True, exist_ok=True)

print(f"Place         : {place_name}")
print(f"Hazard event  : {event}")
print(f"Hazard file   : {hazard_data_path}")
print(f"Cache dir     : {cache_dir}")
print(f"Output dir    : {output_dir}")

Working dir set to: C:\Users\WelJo\IdeaProjects\forschungspraktikum\code
Working dir set to: C:\Users\WelJo\IdeaProjects\forschungspraktikum\code
Working dir set to: C:\Users\WelJo\IdeaProjects\forschungspraktikum\code
Working dir set to: C:\Users\WelJo\IdeaProjects\forschungspraktikum\code
Place         : Trier, Germany
Hazard event  : M
Hazard file   : C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\data\input\Flooding\HazardAreas\nz_hazardArea_fluival_M-DE_cropped_trier.geojson
Cache dir     : C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\data\processed
Output dir    : C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\data\output


## 2. Study-Area Boundary

Loads the pre-processed administrative boundary of Trier from cache
(generated by `system_overview.ipynb`).  Falls back to OSM if the cache
file is absent.

In [3]:
boundary_cache = cache_dir / f"services/boundary_geom_{place_name}.geojson"

if boundary_cache.exists():
    print("Loading boundary from cache …")
    boundary_gdf  = gpd.read_file(boundary_cache)
    boundary_geom = unary_union(boundary_gdf.geometry)
else:
    print("Fetching boundary from OSM …")
    place_gdf     = ox.geocode_to_gdf(place_name)
    boundary_geom = unary_union(place_gdf.geometry)
    gpd.GeoDataFrame(geometry=[boundary_geom], crs="EPSG:4326").to_file(
        boundary_cache, driver="GeoJSON"
    )

minx, miny, maxx, maxy = boundary_geom.bounds
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

print(f"Boundary loaded — centroid ({center_lat:.4f}°N, {center_lon:.4f}°E)")

Loading boundary from cache …
Boundary loaded — centroid (49.7779°N, 6.6495°E)


## 3. HQ100 Flood Data & Stage Interpolation

The `load_or_compute_flood_stages` utility erodes the HQ100 flood polygon
inward in discrete steps to approximate intermediate water levels.  50
stages are pre-computed once and cached as a single GeoJSON file;
subsequent runs skip this step entirely.

Increase `n_stages` (e.g. to 100) for finer transitions at the cost of a
slightly longer first-run computation time.

In [4]:
if not hazard_data_path.exists():
    raise FileNotFoundError(
        f"HQ100 flood data not found:\n  {hazard_data_path}\n"
        "Download the hazard data and place it under "
        "data/input/Flooding/HazardAreas/ as described in the prerequisites."
    )

hq100_gdf = gpd.read_file(hazard_data_path)
print(f"HQ100 source polygons loaded : {len(hq100_gdf)}")

# 50 stages give smooth visual transitions.
# Raise n_stages for finer granularity; lower it for faster first-run.
N_STAGES = 50

stages = load_or_compute_flood_stages(
    cache_dir=cache_dir,
    hq100_gdf=hq100_gdf,
    n_stages=N_STAGES,
    place_name=place_name,
)

non_empty = sum(1 for s in stages if s["geojson"] is not None)
print(f"Flood stages ready           : {len(stages)} total, {non_empty} with polygon")

HQ100 source polygons loaded : 108
Flood stages ready           : 50 total, 49 with polygon


## 4. Simulation Schedule Overview

In [5]:
print("=" * 56)
print("  FLOOD SIMULATION SCHEDULE — 14-day HQ100 event")
print("=" * 56)
print(f"  Total frames  : {SIMULATION_HOURS} (one per hour)")
print(f"  Flood stages  : {len(stages)} pre-computed levels")
print()
print(f"  Phase 1 — Pre-event    : Hours   0–{_RISE_START - 1:3d}  (Days  1–2 )")
print(f"  Phase 2 — Rising water : Hours {_RISE_START:3d}–{_RISE_END  - 1:3d}  (Days  3–6 )")
print(f"  Phase 3 — HQ100 peak   : Hours {_RISE_END:3d}–{_PEAK_END  - 1:3d}  (Days  7–8 )")
print(f"  Phase 4 — Receding     : Hours {_PEAK_END:3d}–{_FALL_END  - 1:3d}  (Days  9–12)")
print(f"  Phase 5 — Post-event   : Hours {_FALL_END:3d}–{SIMULATION_HOURS - 1:3d}  (Days 13–14)")
print("=" * 56)
print()

# Symmetry check
p_h0   = compute_hourly_flood_progress(0)
p_h96  = compute_hourly_flood_progress(96)   # midway through rising phase
p_h168 = compute_hourly_flood_progress(168)  # midway through peak
p_h240 = compute_hourly_flood_progress(240)  # midway through recession
p_h335 = compute_hourly_flood_progress(335)
print("  Symmetry check (rising vs. receding should mirror each other):")
print(f"    Hour   0 (pre)         : {p_h0:.3f}")
print(f"    Hour  96 (mid-rise)    : {p_h96:.3f}")
print(f"    Hour 168 (mid-peak)    : {p_h168:.3f}")
print(f"    Hour 240 (mid-recede)  : {p_h240:.3f}  ← should equal mid-rise")
print(f"    Hour 335 (post)        : {p_h335:.3f}")

  FLOOD SIMULATION SCHEDULE — 14-day HQ100 event
  Total frames  : 336 (one per hour)
  Flood stages  : 50 pre-computed levels

  Phase 1 — Pre-event    : Hours   0– 47  (Days  1–2 )
  Phase 2 — Rising water : Hours  48–143  (Days  3–6 )
  Phase 3 — HQ100 peak   : Hours 144–191  (Days  7–8 )
  Phase 4 — Receding     : Hours 192–287  (Days  9–12)
  Phase 5 — Post-event   : Hours 288–335  (Days 13–14)

  Symmetry check (rising vs. receding should mirror each other):
    Hour   0 (pre)         : 0.000
    Hour  96 (mid-rise)    : 0.500
    Hour 168 (mid-peak)    : 1.000
    Hour 240 (mid-recede)  : 0.500  ← should equal mid-rise
    Hour 335 (post)        : 0.000


## 5. Build & Launch Flood Simulation

Generates a **self-contained HTML animation** and displays it in the
notebook.  All pre-computed flood polygons are embedded directly in the
HTML file — no kernel or server interaction is needed after generation.

The HTML file is also saved to `data/output/flood_simulation.html` and
can be opened directly in any browser for a full-screen presentation.

**In-player controls:**

| Control | Action |
|---|---|
| `► Play` / `⏸ Pause` | Toggle animation |
| `−1h` / `+1h` | Step backward / forward one hour |
| `−1d` / `+1d` | Step backward / forward one day (24 h) |
| **Slider** | Jump to any hour in the simulation |
| **Speed** | 1 / 4 / 8 / 16 / 24 fps |
| `Space` / `K` | Play / Pause (keyboard) |
| `←` / `→` | Step 1 hour (keyboard) |
| `↑` / `↓` | Step 1 day (keyboard) |

In [6]:
html_path = build_flood_animation_html(
    boundary_geom=boundary_geom,
    stages=stages,
    output_path=output_dir / "flood_simulation.html",
    center_lat=center_lat,
    center_lon=center_lon,
)

print(f"Animation file : {html_path}")
print(f"File size      : {html_path.stat().st_size / 1024:.0f} KB")
print()
print("Open the HTML file directly in a browser for a full-screen presentation.")

Animation file : C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\data\output\flood_simulation.html
File size      : 440 KB

Open the HTML file directly in a browser for a full-screen presentation.


In [7]:
import base64

html_bytes = html_path.read_bytes()
html_b64 = base64.b64encode(html_bytes).decode("ascii")

display(HTML(
    f'<iframe src="data:text/html;base64,{html_b64}" width="100%" height="780px" '
    f'frameborder="0" '
    f'style="border-radius:8px; box-shadow:0 2px 14px rgba(0,0,0,0.2);">'
    f'</iframe>'
))

print(f"\nFor full-screen use, open directly in a browser:\n  {html_path}")

C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\.venv\Lib\site-packages\IPython\core\display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")



For full-screen use, open directly in a browser:
  C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\data\output\flood_simulation.html
